# Qwen tree-guided geometry routing benchmark (Colab)

Этот notebook теперь сфокусирован **только на benchmark нового geometry-routed метода** внутри `tree_guided_generate()` для `Qwen/Qwen2.5-1.5B`.

Что делает прогон:
- загружает calibration prompt set
- запускает `tree_guided_generate(..., use_geometry_routing=False)`
- запускает `tree_guided_generate(..., use_geometry_routing=True)`
- считает одинаковый constraint analysis для обоих режимов
- пишет итог в JSON и Markdown
- печатает summary и сами continuation texts по каждому кейсу

Калибровочная часть из этого notebook убрана как временно неактуальная.


> Рекомендуемый runtime: **GPU (T4 / L4 / A100)**.
>
> Notebook гоняет только **tree-guided benchmark**, без отдельного calibration pass.
>
> Для каждого кейса печатаются prompt, legacy continuation и geometry-routed continuation.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest numpy


In [ ]:
%cd /content
import os

if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT
!git fetch --all
!git checkout main
!git pull --ff-only origin main


In [ ]:
import json
from pathlib import Path

import torch

from src.data.qwen_anchor_geometry_cases import make_qwen_anchor_geometry_cases

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 7

cases = make_qwen_anchor_geometry_cases()
print('device =', DEVICE)
print('n_cases =', len(cases))
print('case names =', [case.name for case in cases])


In [ ]:
for case in make_qwen_anchor_geometry_cases():
    print(f'[{case.anchor_class}] {case.name}')
    print(' anchor_group:', case.anchor_group)
    print(' anchor_text :', case.anchor_text)
    print(' prompt      :', case.prompt)
    print()


## Tree-guided benchmark with geometry routing

Здесь сравниваются два режима на одном и том же prompt set:
- legacy: `use_geometry_routing=False`
- geometry-routed: `use_geometry_routing=True`

Артефакты benchmark сохраняются отдельно в JSON и Markdown.


In [ ]:
from collections import Counter
from dataclasses import replace
from datetime import UTC, datetime

from src.model.config import TOY_CONFIG
from src.model.qwen_anchor_overlay import QwenAnchorOverlay
from scripts.run_qwen_geometry_generation_calibration import KEYWORD_MAP, compute_constraint_analysis

TREE_BENCHMARK_MAX_NEW_TOKENS = 120
TREE_BENCHMARK_MAX_LENGTH = 256
TREE_BENCHMARK_TEMPERATURE = 0.7
TREE_BENCHMARK_CHUNK_SIZE = 20
TREE_BENCHMARK_TREE_CHECK_INTERVAL = 10
TREE_BENCHMARK_MAX_REVISIONS = 3

OUTPUT_TREE_BENCH_JSON = '/content/ABPT/archive/qwen_tree_guided_geometry_benchmark.json'
OUTPUT_TREE_BENCH_MD = '/content/ABPT/docs/research/qwen_tree_guided_geometry_benchmark.md'

TREE_CFG = replace(
    TOY_CONFIG,
    anchor_threshold=0.10,
    anchor_revision_threshold=0.35,
    anchor_contradiction_threshold=0.20,
    anchor_dead_end_threshold=0.50,
)

tree_overlay = QwenAnchorOverlay.from_pretrained(
    model_name=MODEL_NAME,
    cfg=TREE_CFG,
    device=DEVICE,
    torch_dtype=torch.float16 if DEVICE.startswith('cuda') else None,
    low_cpu_mem_usage=True,
)
tree_overlay.eval()

print('tree benchmark model loaded on', DEVICE)
print('output_tree_json =', OUTPUT_TREE_BENCH_JSON)
print('output_tree_md =', OUTPUT_TREE_BENCH_MD)


In [ ]:
def tree_case_analysis(case, text):
    keyword_spec = KEYWORD_MAP.get(case.anchor_group)
    if keyword_spec is None:
        return {
            'constraint_score': None,
            'constraint_satisfied': False,
            'degenerate_output': False,
            'drift_detected': False,
        }
    return compute_constraint_analysis(text, keyword_spec)


tree_benchmark_rows = []
for case in make_qwen_anchor_geometry_cases():
    torch.manual_seed(SEED)
    legacy = tree_overlay.tree_guided_generate(
        prompt=case.prompt,
        max_new_tokens=TREE_BENCHMARK_MAX_NEW_TOKENS,
        max_length=TREE_BENCHMARK_MAX_LENGTH,
        temperature=TREE_BENCHMARK_TEMPERATURE,
        chunk_size=TREE_BENCHMARK_CHUNK_SIZE,
        tree_check_interval=TREE_BENCHMARK_TREE_CHECK_INTERVAL,
        max_revisions=TREE_BENCHMARK_MAX_REVISIONS,
        use_geometry_routing=False,
    )

    torch.manual_seed(SEED)
    routed = tree_overlay.tree_guided_generate(
        prompt=case.prompt,
        max_new_tokens=TREE_BENCHMARK_MAX_NEW_TOKENS,
        max_length=TREE_BENCHMARK_MAX_LENGTH,
        temperature=TREE_BENCHMARK_TEMPERATURE,
        chunk_size=TREE_BENCHMARK_CHUNK_SIZE,
        tree_check_interval=TREE_BENCHMARK_TREE_CHECK_INTERVAL,
        max_revisions=TREE_BENCHMARK_MAX_REVISIONS,
        use_geometry_routing=True,
    )

    legacy_analysis = tree_case_analysis(case, legacy['continuation_text'])
    routed_analysis = tree_case_analysis(case, routed['continuation_text'])
    row = {
        'name': case.name,
        'anchor_group': case.anchor_group,
        'prompt': case.prompt,
        'legacy_generation_mode': legacy.get('generation_mode', 'tree_guided'),
        'legacy_constraint_score': legacy_analysis['constraint_score'],
        'legacy_constraint_satisfied': legacy_analysis['constraint_satisfied'],
        'legacy_revision_count': legacy.get('revision_count', 0),
        'legacy_continuation_text': legacy['continuation_text'],
        'routed_generation_mode': routed.get('generation_mode', 'unknown'),
        'routed_cluster': routed.get('geometry_route', {}).get('cluster', 'unknown'),
        'routed_route': routed.get('geometry_route', {}).get('route', 'unknown'),
        'routed_constraint_score': routed_analysis['constraint_score'],
        'routed_constraint_satisfied': routed_analysis['constraint_satisfied'],
        'routed_revision_count': routed.get('revision_count', 0),
        'routed_continuation_text': routed['continuation_text'],
        'delta': float((routed_analysis['constraint_score'] or 0.0) - (legacy_analysis['constraint_score'] or 0.0)),
    }
    tree_benchmark_rows.append(row)
    print(
        f"{case.name}: legacy={row['legacy_constraint_score']} | "
        f"routed={row['routed_constraint_score']} | "
        f"delta={row['delta']} | mode={row['routed_generation_mode']} | "
        f"cluster={row['routed_cluster']}"
    )
    print('prompt:')
    print(row['prompt'])
    print('legacy continuation:')
    print(row['legacy_continuation_text'])
    print('routed continuation:')
    print(row['routed_continuation_text'])
    print('-' * 100)


In [ ]:
legacy_scores = [float(row['legacy_constraint_score'] or 0.0) for row in tree_benchmark_rows]
routed_scores = [float(row['routed_constraint_score'] or 0.0) for row in tree_benchmark_rows]
mode_counts = Counter(row['routed_generation_mode'] for row in tree_benchmark_rows)
cluster_counts = Counter(row['routed_cluster'] for row in tree_benchmark_rows)
route_counts = Counter(row['routed_route'] for row in tree_benchmark_rows)

wins = sum(1 for row in tree_benchmark_rows if row['delta'] > 0)
losses = sum(1 for row in tree_benchmark_rows if row['delta'] < 0)
ties = sum(1 for row in tree_benchmark_rows if row['delta'] == 0)

summary = {
    'n_cases': len(tree_benchmark_rows),
    'legacy_mean_constraint_score': float(sum(legacy_scores) / len(legacy_scores)) if legacy_scores else None,
    'routed_mean_constraint_score': float(sum(routed_scores) / len(routed_scores)) if routed_scores else None,
    'delta_vs_legacy': float((sum(routed_scores) - sum(legacy_scores)) / len(routed_scores)) if routed_scores else None,
    'wins': wins,
    'losses': losses,
    'ties': ties,
    'mode_counts': dict(mode_counts),
    'cluster_counts': dict(cluster_counts),
    'route_counts': dict(route_counts),
}

payload_tree_bench = {
    'metadata': {
        'created_at_utc': datetime.now(UTC).isoformat(),
        'model_name': MODEL_NAME,
        'device': DEVICE,
        'max_new_tokens': TREE_BENCHMARK_MAX_NEW_TOKENS,
        'max_length': TREE_BENCHMARK_MAX_LENGTH,
        'temperature': TREE_BENCHMARK_TEMPERATURE,
        'chunk_size': TREE_BENCHMARK_CHUNK_SIZE,
        'tree_check_interval': TREE_BENCHMARK_TREE_CHECK_INTERVAL,
        'max_revisions': TREE_BENCHMARK_MAX_REVISIONS,
        'seed': SEED,
    },
    'results': tree_benchmark_rows,
    'summary': summary,
}

report_lines = [
    '# Qwen Tree-Guided Geometry Routing Benchmark',
    '',
    '## Summary',
    '',
    f"- Date: {datetime.now(UTC).strftime('%Y-%m-%d %H:%M UTC')}",
    f"- Model: `{MODEL_NAME}`",
    f"- Device: `{DEVICE}`",
    f"- n_cases: `{summary['n_cases']}`",
    f"- legacy_mean_constraint_score: `{summary['legacy_mean_constraint_score']:.3f}`",
    f"- routed_mean_constraint_score: `{summary['routed_mean_constraint_score']:.3f}`",
    f"- delta_vs_legacy: `{summary['delta_vs_legacy']:.3f}`",
    f"- wins: `{summary['wins']}`",
    f"- losses: `{summary['losses']}`",
    f"- ties: `{summary['ties']}`",
    f"- mode_counts: `{summary['mode_counts']}`",
    f"- cluster_counts: `{summary['cluster_counts']}`",
    f"- route_counts: `{summary['route_counts']}`",
    '',
    '## Per-case table',
    '',
    '| name | legacy_score | routed_score | delta | routed_mode | routed_cluster | routed_route |',
    '| --- | ---: | ---: | ---: | --- | --- | --- |',
]
for row in tree_benchmark_rows:
    report_lines.append(
        '| ' + ' | '.join([
            row['name'],
            f"{float(row['legacy_constraint_score'] or 0.0):.3f}",
            f"{float(row['routed_constraint_score'] or 0.0):.3f}",
            f"{float(row['delta']):.3f}",
            row['routed_generation_mode'],
            row['routed_cluster'],
            row['routed_route'],
        ]) + ' |'
    )

report_lines.extend(['', '## Per-case generations', ''])
for row in tree_benchmark_rows:
    report_lines.extend([
        f"### {row['name']}",
        '',
        f"- routed_mode: `{row['routed_generation_mode']}`",
        f"- routed_cluster: `{row['routed_cluster']}`",
        f"- routed_route: `{row['routed_route']}`",
        f"- legacy_score: `{float(row['legacy_constraint_score'] or 0.0):.3f}`",
        f"- routed_score: `{float(row['routed_constraint_score'] or 0.0):.3f}`",
        f"- delta: `{float(row['delta']):.3f}`",
        '',
        '**Prompt**',
        '',
        '```text',
        row['prompt'],
        '```',
        '',
        '**Legacy continuation**',
        '',
        '```text',
        row['legacy_continuation_text'],
        '```',
        '',
        '**Geometry-routed continuation**',
        '',
        '```text',
        row['routed_continuation_text'],
        '```',
        '',
    ])
report_text = '\n'.join(report_lines) + '\n'

Path(OUTPUT_TREE_BENCH_JSON).write_text(json.dumps(payload_tree_bench, ensure_ascii=False, indent=2), encoding='utf-8')
Path(OUTPUT_TREE_BENCH_MD).write_text(report_text, encoding='utf-8')

print('tree benchmark summary:')
print(summary)
print()
print(report_text)


In [ ]:
# Optional: download artifacts from Colab runtime
# from google.colab import files
# files.download(OUTPUT_TREE_BENCH_JSON)
# files.download(OUTPUT_TREE_BENCH_MD)
